In [1]:
from pathlib import Path
import duckdb

In [2]:
# Initialize data Path & database file
data_path = Path('../data/')
db_file = '../olist_medallion_architecture.db'

# Connect to database file
conn = duckdb.connect(db_file)

### Build bronze layer objects
* Simple views w/o any changes to the underlying raw data

In [3]:
# Terms for removal
remove_set = set(['olist', 'order', 'dataset'])

# List raw Olist CSV file names
for p in data_path.glob('*.csv'):
    # Remove terms from file name
    file_name_list = p.stem.split('_')

    # Simplify table name from file name
    view_name_list = [x for x in file_name_list if x not in remove_set]
    view_name = '_'.join(view_name_list)

    # Special case: 
    if view_name == 'items':
        view_name = 'order_items'

    # Create bronze views
    print(f'Creating bronze_{view_name}')
    conn.execute(f"""
        CREATE OR REPLACE VIEW bronze_{view_name} AS 
        SELECT * FROM read_csv_auto('{data_path}/{p.name}');
    """)

print(f'\nBronze layer views created...')

Creating bronze_brazil_state_code_ref
Creating bronze_customers
Creating bronze_geolocation
Creating bronze_orders
Creating bronze_order_items
Creating bronze_payments
Creating bronze_reviews
Creating bronze_products
Creating bronze_sellers
Creating bronze_product_category_name_translation

Bronze layer views created...


### Build silver layer views from bronze views
* CAST all date/timestamp columns explicitly as timestamp
* Simplify columns names
* Deduplicate and clean data
* Join views using primary and foreign key relations

In [14]:
conn.query(""" 
    SELECT
        review_id,
        review_comment_title,
        translate_to_english(review_comment_title) AS title,
        review_comment_message
    FROM bronze_reviews
    WHERE review_comment_message IS NOT NULL
    LIMIT 5;
""")

text: recomendo - <class 'str'>


InvalidInputException: Invalid Input Error: Python exception occurred while executing the UDF: TooManyRequests: Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function

text: recomendo - <class 'str'>


InvalidInputException: Invalid Input Error: Python exception occurred while executing the UDF: TooManyRequests: Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function

In [ ]:
from transformers import pipeline
from duckdb.sqltypes import VARCHAR
import time

# Initialize Portuguese-to-English translation pipeline
translator_pipe = pipeline('translation', model = 'Helsinki-NLP/opus-mt-pt-en')

# User-defined functions (UDF) definitions 
def translate_to_english_udf (text: str) -> str:
    if not text:
        return ''
    try:
        en_text = translator_pipe(text, )
    
    print(f'text: {text} - {type(text)}')
    if text:
        text = MyMemoryTranslator(source = 'pt-PT', target = 'en-US').translate(text)
        time.sleep(1.5)
    return text

# Register UDF in duckdb
try:
    conn.remove_function('translate_to_english')
except Exception:
    pass

conn.create_function('translate_to_english', translate_to_english, [VARCHAR], VARCHAR)

In [ ]:
import time

time.sleep

In [6]:
# Olist reviews table definition (w/ UDF to translate title and comment into English)
conn.execute("""
    CREATE OR REPLACE TABLE silver_reviews AS
        SELECT
            review_id,
            order_id,
            CAST(review_creation_date AS TIMESTAMP) AS creation_timestamp,
            CAST(review_answer_timestamp AS TIMESTAMP) AS answer_timestamp,
            review_score AS rating,
            translate_to_english(review_comment_title) AS title,
            translate_to_english(review_comment_message) AS comment
        FROM bronze_reviews;
""")

InvalidInputException: Invalid Input Error: Python exception occurred while executing the UDF: TooManyRequests: Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function

In [ ]:
# Olist order view definitions
conn.execute("""
    CREATE OR REPLACE VIEW silver_order_items AS
        SELECT
            order_id,
            order_item_id,
            product_id,
            CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_timestamp,
            price AS item_price,
            freight_value AS shipping_cost
        FROM bronze_order_items;
""")
conn.execute("""
    CREATE OR REPLACE VIEW silver_orders AS
        SELECT
            order_id,
            customer_id,
            order_status,
            CAST(order_purchase_timestamp AS TIMESTAMP) AS purchase_timestamp,
            CAST(order_approved_at AS TIMESTAMP) AS approved_timestamp,
            CAST(order_delivered_carrier_date AS TIMESTAMP) AS delivered_carrier_timestamp,
            CAST(order_delivered_customer_date AS TIMESTAMP) AS delivered_customer_timestamp,
            CAST(order_estimated_delivery_date AS TIMESTAMP) AS estimated_delivery_timestamp
        FROM bronze_orders;
""")

conn.execute("""
    CREATE OR REPLACE VIEW silver_payments AS
        SELECT
            order_id,
            payment_sequential AS payment_sequence,
            payment_type AS type,
            payment_installments AS installments,
            payment_value AS value
        FROM bronze_payments;
""")

# Olist reviews table definition (w/ UDF to translate title and comment into English)
conn.execute("""
    CREATE OR REPLACE TABLE silver_reviews AS
        SELECT
            review_id,
            order_id,
            CAST(review_creation_date AS TIMESTAMP) AS creation_timestamp,
            CAST(review_answer_timestamp AS TIMESTAMP) AS answer_timestamp,
            review_score AS rating,
            translate_to_english(review_comment_title) AS title,
            translate_to_english(review_comment_message) AS comment
        FROM bronze_reviews;
""")

# Olist products view definition (w/ english category name)
conn.execute("""
    CREATE OR REPLACE VIEW silver_products AS
        SELECT
            p.product_id,
            COALESCE(pt.product_category_name_english, p.product_category_name) AS category,
            p.product_name_lenght AS name_length,
            p.product_description_lenght AS description_length,
            p.product_photos_qty AS photo_qty,
            p.product_weight_g AS weight_g,
            p.product_length_cm AS length_cm,
            p.product_height_cm AS height_cm,
            p.product_width_cm AS width_cm
        FROM bronze_products p
        LEFT JOIN bronze_product_category_name_translation pt 
            ON p.product_category_name = pt.product_category_name;
""")

# Olist customers & sellers view definitions (w/ geolocation & state name)
conn.execute("""
    CREATE OR REPLACE VIEW silver_customers AS
        SELECT
            customer_id,
            customer_unique_id AS shopper_id,
            customer_zip_code_prefix AS zip_code,
            customer_city AS city,
            customer_state AS state_code,
            bs.state_name,
            geolocation_lat AS latitude,
            geolocation_lng AS longitude
        FROM bronze_customers c
        LEFT JOIN bronze_geolocation g 
            ON c.customer_zip_code_prefix = g.geolocation_zip_code_prefix
        LEFT JOIN bronze_brazil_state_code_ref bs 
            ON c.customer_state = bs.state_code;
""")

conn.execute("""
    CREATE OR REPLACE VIEW silver_sellers AS
        SELECT
            seller_id,
            seller_zip_code_prefix AS zip_code,
            seller_city AS city,
            seller_state AS state_code,
            bs.state_name,
            geolocation_lat AS latitude,
            geolocation_lng AS longitude
        FROM bronze_sellers s
        LEFT JOIN bronze_geolocation g 
            ON s.seller_zip_code_prefix = g.geolocation_zip_code_prefix
        LEFT JOIN bronze_brazil_state_code_ref bs 
            ON s.seller_state = bs.state_code;
""")

print(f'\nSilver layer views created...')


Silver layer views created...


In [9]:
# Close database connection
conn.close()